In [1]:
import pandas as pd
import re
from pathlib import Path

In [2]:
base_path = Path("../../../daos-benchmarking/ior_results")

def extract_segments(dir_name):
    m = re.search(r'seg-(\d+)', dir_name)
    return int(m.group(1)) if m else None

def extract_filename_meta(stem):
    """Extract nnodes, ppn, xfer_str from stems like write_n-8_ppn-16_tx-2M or rw_n-8_ppn-16_tx-2M."""
    m = re.match(r'^(?:write|rw|read)_n-(\d+)_ppn-(\d+)_tx-(\w+)$', stem)
    if m:
        return int(m.group(1)), int(m.group(2)), m.group(3)
    return None, None, None

records = []
skipped = []
errors  = []

for csv_file in sorted(base_path.rglob("*.csv")):
    # Skip root-level aggregate CSVs (ior_rw.csv, ior_all_results.csv, etc.)
    if csv_file.parent == base_path:
        continue

    rel_parts = csv_file.relative_to(base_path).parts
    category       = rel_parts[0]
    experiment_run = rel_parts[1] if len(rel_parts) > 2 else ""

    segments = extract_segments(experiment_run)
    nnodes, ppn, xfer_str = extract_filename_meta(csv_file.stem)

    if nnodes is None:
        skipped.append(str(csv_file.relative_to(base_path)))
        continue

    try:
        df = pd.read_csv(csv_file)
        if df.empty:
            errors.append(f"empty: {csv_file.relative_to(base_path)}")
            continue
    except Exception as e:
        errors.append(f"read error ({e}): {csv_file.relative_to(base_path)}")
        continue

    df["fromCSV"]         = csv_file.name
    df["source_category"] = category
    df["experiment_run"]  = experiment_run
    df["segments"]        = segments
    df["nnodes"]          = nnodes
    df["tasks_per_node"]  = ppn
    df["xfer_str"]        = xfer_str
    df["bw(GiB/s)"]       = df["bw(MiB/s)"] / 1024
    df["Latency(ms)"]     = df["Latency"] * 1000

    records.append(df)

all_df = pd.concat(records, ignore_index=True)

print(f"Total rows   : {len(all_df)}")
print(f"Source files : {all_df['fromCSV'].nunique()}")
if errors:
    print(f"\nWarnings ({len(errors)} files):")
    for e in errors:
        print(f"  {e}")
if skipped:
    print(f"\nSkipped — unrecognised name pattern ({len(skipped)} files):")
    for s in skipped:
        print(f"  {s}")

Total rows   : 4697
Source files : 449

Warnings (1 files):
  read error (No columns to parse from file): ior_results_archive/ior-results-rw_n-8_seg-32_20251006_11-39-32/rw_n-8_ppn-120_tx-2M.csv

Skipped — unrecognised name pattern (4 files):
  ior_results_archive/ior_results.csv
  ior_results_xmei/merged_ior_results_n-16_48core-skip1.csv
  ior_results_xmei/merged_ior_results_n-1_24core-skip4.csv
  ior_results_xmei/merged_ior_results_n-1_48core-skip1.csv


In [3]:
print("Rows per source_category:")
print(all_df["source_category"].value_counts().to_string())
print("\nRows per nnodes:")
print(all_df["nnodes"].value_counts().sort_index().to_string())
print("\nRows per xfer_str:")
print(all_df["xfer_str"].value_counts().sort_index().to_string())
print("\nRows per access:")
print(all_df["access"].value_counts().to_string())
all_df.head(10)

Rows per source_category:
source_category
ior_results_archive    1711
write_then_read        1196
ior_results_xmei       1190
write_only              600

Rows per nnodes:
nnodes
1     1685
2      460
4      460
8     1196
16     600
32     296

Rows per xfer_str:
xfer_str
16K    1075
16M    1070
1M     1071
2M     1061
8M      420

Rows per access:
access
write    3856
read      841


,access,bw(MiB/s),IOPS,Latency,block(KiB),xfer(KiB),open(s),wr/rd(s),close(s),total(s),...,iter,fromCSV,source_category,experiment_run,segments,nnodes,tasks_per_node,xfer_str,bw(GiB/s),Latency(ms)
0,write,11051.5793,707418.8816,0.0423,131072.0,16.0,0.1821,355.7415,15.3104,355.8007,...,0,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,10.792558,42.3
1,read,10040.5356,642599.4887,0.0476,131072.0,16.0,0.1018,391.6253,22.8077,391.6285,...,0,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,9.805211,47.6
2,write,11073.3765,708757.7958,0.0424,131072.0,16.0,0.1603,355.0694,14.8552,355.1004,...,1,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,10.813844,42.4
3,read,10033.3489,642139.4703,0.0468,131072.0,16.0,0.1132,391.9059,22.7446,391.9090,...,1,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,9.798192,46.8
4,write,11042.5184,706846.1071,0.0420,131072.0,16.0,0.1602,356.0297,15.3187,356.0927,...,2,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,10.783709,42.0
5,read,10029.3590,641884.1751,0.0468,131072.0,16.0,0.1314,392.0618,20.9410,392.0649,...,2,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,9.794296,46.8
6,write,11044.8422,706899.4420,0.0418,131072.0,16.0,0.1416,356.0029,15.4852,356.0178,...,3,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,10.785979,41.8
7,read,9992.1229,639500.9168,0.0468,131072.0,16.0,0.0991,393.5229,21.0925,393.5260,...,3,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,9.757933,46.8
8,write,11016.5124,705157.2537,0.0416,131072.0,16.0,0.1473,356.8824,15.7940,356.9333,...,4,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,10.758313,41.6
9,read,9950.2841,636823.0961,0.0464,131072.0,16.0,0.1392,395.1776,21.1633,395.1807,...,4,rw_n-8_ppn-120_tx-16K.csv,ior_results_archive,ior-results-rw_n-8_seg-32_20251006_11-39-32,32,8,120,16K,9.717074,46.4


In [4]:
output_csv = Path("ior_all_results.csv")
all_df.to_csv(output_csv)
print(f"Saved to : {output_csv.resolve()}")
print(f"Shape    : {all_df.shape}")
print(f"Columns  : {list(all_df.columns)}")

Saved to : /Users/xmei/Documents/LDRD/pdsw26_daos-bench-res/results/aurora/ior/ior_all_results.csv
Shape    : (4697, 21)
Columns  : ['access', 'bw(MiB/s)', 'IOPS', 'Latency', 'block(KiB)', 'xfer(KiB)', 'open(s)', 'wr/rd(s)', 'close(s)', 'total(s)', 'numTasks', 'iter', 'fromCSV', 'source_category', 'experiment_run', 'segments', 'nnodes', 'tasks_per_node', 'xfer_str', 'bw(GiB/s)', 'Latency(ms)']


In [5]:
# --- Config/run summary ---
# A "config"  = unique (nnodes, tasks_per_node, xfer_str, access, segments)
# A "run"     = unique experiment_run directory (one job submission)
# An "iter"   = one repetition within a run (iter 0–4)

config_cols = ["nnodes", "tasks_per_node", "xfer_str", "access", "segments"]

summary = (
    all_df.groupby(config_cols)
    .agg(
        n_runs=("experiment_run", "nunique"),
        n_iters=("iter", "count"),          # total rows = runs × iters_per_run
    )
    .reset_index()
    .sort_values(config_cols)
)

print(f"Unique configs : {len(summary)}")
print(f"Total rows     : {summary['n_iters'].sum()}")
print()
print(summary.to_string(index=False))

Unique configs : 553
Total rows     : 4697

 nnodes  tasks_per_node xfer_str access  segments  n_runs  n_iters
      1               6      16K   read        32       1        5
      1               6      16K  write        32       3       15
      1               6      16M   read        32       1        5
      1               6      16M  write        32       3       15
      1               6       1M   read        32       1        5
      1               6       1M  write        32       3       15
      1               6       2M   read        32       1        5
      1               6       2M  write        32       3       15
      1               6       8M  write        32       1        5
      1               8      16K   read        32       1        5
      1               8      16K  write         1       1        5
      1               8      16K  write         6       1        5
      1               8      16K  write        32       6       30
      1           

In [6]:
# --- Run-to-run variability: xfer 2M & 16M, segments=32, ppn in [8,16,32,48] ---
config_cols = ["nnodes", "tasks_per_node", "xfer_str", "access", "segments"]
display_cols = ["nnodes", "tasks_per_node", "xfer_str",
                "n_runs", "mean", "min", "max", "spread_GiB/s",
                "cv_within_pct", "cv_across"]

sub = all_df[
    all_df["xfer_str"].isin(["2M", "16M"]) &
    (all_df["segments"] == 32) &
    all_df["tasks_per_node"].isin([8, 16, 32, 48])
].copy()

# ── within-run CV ───────────────────────────────────────────────────────────
within = (
    sub.groupby(config_cols + ["experiment_run"])["bw(GiB/s)"]
    .agg(mean="mean", std="std")
    .assign(cv_within=lambda d: d["std"] / d["mean"] * 100)
    .reset_index()
    .groupby(config_cols)["cv_within"]
    .mean()
    .rename("cv_within_pct")
    .reset_index()
)

# ── across-run CV ───────────────────────────────────────────────────────────
run_means = (
    sub.groupby(config_cols + ["experiment_run"])["bw(GiB/s)"]
    .mean()
    .reset_index(name="run_mean")
)
across = (
    run_means.groupby(config_cols)["run_mean"]
    .agg(n_runs="count", mean="mean", std="std",
         min="min", max="max",
         cv_across=lambda x: x.std() / x.mean() * 100 if len(x) > 1 else float("nan"))
    .reset_index()
)

stats = across.merge(within, on=config_cols).sort_values(config_cols)
stats["spread_GiB/s"] = stats["max"] - stats["min"]

pd.set_option("display.float_format", "{:.1f}".format)
pd.set_option("display.max_rows", 200)

print("=== WRITE ===")
display(stats[stats["access"] == "write"][display_cols].reset_index(drop=True))

print("=== READ ===")
display(stats[stats["access"] == "read"][display_cols].reset_index(drop=True))

=== WRITE ===


,nnodes,tasks_per_node,xfer_str,n_runs,mean,min,max,spread_GiB/s,cv_within_pct,cv_across
0,1,8,16M,6,44.0,40.6,45.9,5.3,2.6,4.4
1,1,8,2M,6,21.2,18.4,22.1,3.7,4.8,6.6
2,1,16,16M,6,70.4,42.4,79.3,36.9,2.9,20.3
3,1,16,2M,6,38.1,22.2,44.3,22.1,5.3,22.3
4,1,32,16M,6,96.2,52.3,124.0,71.7,5.6,27.9
5,1,32,2M,6,60.6,27.9,78.5,50.6,4.4,34.4
6,1,48,16M,6,103.1,56.2,124.7,68.5,3.0,24.2
7,1,48,2M,6,80.3,31.5,100.5,69.0,4.0,34.5
8,2,8,16M,2,84.4,78.3,90.4,12.1,4.2,10.2
9,2,8,2M,2,40.3,38.6,42.0,3.4,4.3,6.0


=== READ ===


,nnodes,tasks_per_node,xfer_str,n_runs,mean,min,max,spread_GiB/s,cv_within_pct,cv_across
0,1,8,16M,1,58.1,58.1,58.1,0.0,1.7,NaN
1,1,8,2M,1,26.1,26.1,26.1,0.0,1.7,NaN
2,1,16,16M,1,41.2,41.2,41.2,0.0,6.3,NaN
3,1,16,2M,1,47.7,47.7,47.7,0.0,2.9,NaN
4,1,32,16M,1,30.9,30.9,30.9,0.0,4.0,NaN
5,1,32,2M,1,47.3,47.3,47.3,0.0,4.5,NaN
6,1,48,16M,1,32.7,32.7,32.7,0.0,6.8,NaN
7,1,48,2M,1,44.5,44.5,44.5,0.0,1.5,NaN
8,2,8,16M,1,114.8,114.8,114.8,0.0,2.1,NaN
9,2,8,2M,1,50.2,50.2,50.2,0.0,4.4,NaN


In [7]:
# --- Within-run stability (iter-to-iter CV per run) ---
within_detail = (
    sub.groupby(config_cols + ["experiment_run"])["bw(GiB/s)"]
    .agg(mean="mean", std="std", n_iters="count")
    .assign(cv_within=lambda d: d["std"] / d["mean"] * 100)
    .reset_index()
)

pd.set_option("display.float_format", "{:.1f}".format)
for acc in ["write", "read"]:
    t = within_detail[within_detail["access"] == acc]["cv_within"]
    print(f"=== {acc.upper()} — cv_within per run ({len(t)} runs) ===")
    print(t.describe().rename({
        "count": "n_runs", "mean": "mean_cv", "std": "std_cv",
        "min": "min_cv", "25%": "Q1", "50%": "median", "75%": "Q3", "max": "max_cv"
    }).to_string())
    print(f"  cv > 10% : {(t > 10).sum()} runs ({(t > 10).mean()*100:.0f}%)")
    print(f"  cv > 20% : {(t > 20).sum()} runs ({(t > 20).mean()*100:.0f}%)")
    print()

    worst = (
        within_detail[within_detail["access"] == acc]
        .nlargest(8, "cv_within")
        [["nnodes", "tasks_per_node", "xfer_str", "experiment_run", "n_iters", "mean", "cv_within"]]
    )
    print(f"  Worst 8 runs by cv_within:")
    display(worst.reset_index(drop=True))
    print()

=== WRITE — cv_within per run (152 runs) ===
n_runs    152.0
mean_cv     9.8
std_cv     12.0
min_cv      0.7
Q1          3.2
median      5.6
Q3         11.0
max_cv     74.9
  cv > 10% : 43 runs (28%)
  cv > 20% : 18 runs (12%)

  Worst 8 runs by cv_within:


,nnodes,tasks_per_node,xfer_str,experiment_run,n_iters,mean,cv_within
0,32,8,2M,ior-results-w_n-32_seg-32_8304643,5,88.5,74.9
1,2,16,2M,ior-results-w_n-2_seg-32_8305889,5,45.7,72.7
2,16,48,2M,ior-results-w_n-16_seg-32_8304613,5,200.2,50.8
3,32,32,16M,ior-results-rw_n-32_seg-32_8378205,5,1104.6,44.6
4,16,32,2M,ior-results-w_n-16_seg-32_8304613,5,118.2,43.7
5,32,48,2M,ior-results-rw_n-32_seg-32_8378205,3,654.5,42.3
6,16,48,2M,ior-results-w_n-16_seg-32_20251017_48core-skip1,5,933.9,35.8
7,32,32,2M,ior-results-rw_n-32_seg-32_8378205,5,911.9,34.6



=== READ — cv_within per run (56 runs) ===
n_runs    56.0
mean_cv    8.3
std_cv     7.8
min_cv     0.5
Q1         2.9
median     6.3
Q3        10.2
max_cv    39.5
  cv > 10% : 15 runs (27%)
  cv > 20% : 4 runs (7%)

  Worst 8 runs by cv_within:


,nnodes,tasks_per_node,xfer_str,experiment_run,n_iters,mean,cv_within
0,32,16,16M,ior-results-rw_n-32_seg-32_8378205,5,1427.7,39.5
1,16,16,16M,ior-results-rw_n-16_seg-32_8340995,5,443.8,31.1
2,32,48,2M,ior-results-rw_n-32_seg-32_8378205,3,1602.0,28.7
3,32,32,2M,ior-results-rw_n-32_seg-32_8378205,5,1541.3,20.3
4,16,32,16M,ior-results-rw_n-16_seg-32_8340995,5,237.6,19.0
5,8,32,16M,ior-results-rw_n-8_seg-32_8305849,5,146.7,17.9
6,4,48,16M,ior-results-rw_n-4_seg-32_8330848,5,99.3,16.3
7,4,16,16M,ior-results-rw_n-4_seg-32_8330848,5,117.7,15.2


### Run-to-run variability: xfer 2 MiB & 16 MiB (seg=32, ppn 8/16/32/48)

Two levels of variance are reported:
- **`cv_within_pct`** — CV (std/mean × 100%) across the 5 iterations *within* a single run. Reflects measurement noise inside one job submission.
- **`cv_across`** — CV of per-run means across independent job submissions. Reflects true run-to-run reproducibility (NaN when only one run exists).

#### Within-run stability (iter-to-iter)

**Write (152 individual runs):**
- Median `cv_within` = 5.6% — the typical run is reasonably stable.
- But 43 / 152 runs (28%) exceed 10%, and 18 / 152 (12%) exceed 20%.
- The worst offenders are concentrated at large node counts: n=32/ppn=8/2M reaches 75%, n=2/ppn=16/2M reaches 73%, and several n=16/n=32 runs exceed 30–50%.
- Small configs (n ≤ 4) are consistently stable within a run (cv_within < 10%).

**Read (56 individual runs):**
- Median `cv_within` = 6.3% — similar to write overall.
- 15 / 56 runs (27%) exceed 10%, 4 / 56 (7%) exceed 20%.
- Worst cases: n=32/ppn=16/16M (40%), n=16/ppn=16/16M (31%), n=32/ppn=48/2M (29%).
- Again, instability is driven by large node counts; small configs are stable.

**Overall:** Within-run stability is *not* uniformly good. The 5 iterations within a single run can vary substantially at n ≥ 16, meaning even a single run's mean is an unreliable summary for those configs.

#### Across-run variability (run-to-run)

##### Write (48 configs — all have ≥ 2 runs)

| Nodes | `cv_across` range | Notes |
|------:|------------------:|-------|
| 1     | 4–35%             | ppn=8 stable (4–7%); variability increases with ppn |
| 2     | 2–42%             | Mostly stable; n=2/ppn=16/2M outlier (42%) |
| 4     | 2–23%             | Stable at ppn=8/48; n=4/ppn=32 reaches 23% |
| 8     | 8–51%             | ppn=8 stable (8–9%); ppn=32/48 highly variable (47–51%) |
| 16    | 33–73%            | Highly unstable across all ppn — 2–4× swings between runs |
| 32    | 66–110%           | Extremely unstable — bandwidth can differ by ~10× between runs |

##### Read (48 configs — mostly single-run)

- **n=1 to n=4:** Only 1 run per config — `cv_across` unavailable. Within-run CV is low (< 10%).
- **n=8:** 2 runs; `cv_across` ranges 10–57%. ppn=8 is most stable (~10%); ppn=32/48 reach 43–57%.
- **n=16 and n=32:** Only 1 run per config — reproducibility unknown. Within-run CV can be high (up to 40% for n=32/ppn=16/16M).

#### Summary

Neither within-run nor across-run stability can be assumed at large node counts (n ≥ 16). The 5 iterations inside a single run already show large swings, and independent job submissions diverge even more. Reliable conclusions require multiple runs at n ≥ 8 with high ppn, and especially at n ≥ 16.